# LintGate Golden Path Pipeline — Wayfinder

Full specification coverage pipeline: **Sweep → Analyze → Reconcile → Report**

Runs LintGate's mutation profiling + spec analysis on the Wayfinder codebase.

**Runtime > Run all.** ~5-10 min.

## Step 1: Clone & install


In [ ]:
import importlib, os, shutil, subprocess, sys

LINTGATE_URL = "https://github.com/rohanvinaik/LintGate.git"
WAYFINDER_URL = "https://github.com/rohanvinaik/Wayfinder.git"
BRANCH = "main"
LINTGATE_DIR = "/content/lintgate"
PROJECT_DIR = "/content/project"
SRC_DIRS = ['src']

# Uncomment if repos are private:
# GITHUB_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"

lintgate_clone = LINTGATE_URL
wayfinder_clone = WAYFINDER_URL
try:
    GITHUB_TOKEN
    lintgate_clone = LINTGATE_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    wayfinder_clone = WAYFINDER_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    print('Using authenticated clone')
except NameError:
    print('Using public clone')

# 1. Clone and install LintGate (the tool)
if os.path.exists(LINTGATE_DIR):
    shutil.rmtree(LINTGATE_DIR)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', lintgate_clone, LINTGATE_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    raise RuntimeError(f'LintGate clone failed: {result.stderr}')
print(f'Cloned LintGate to {LINTGATE_DIR}')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'hatchling', 'pyyaml', 'packaging'],
    check=True
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', LINTGATE_DIR],
    capture_output=True, text=True
)

# 2. Clone Wayfinder (the target project)
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, wayfinder_clone, PROJECT_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', wayfinder_clone, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f'Wayfinder clone failed: {result.stderr}')
print(f'Cloned Wayfinder to {PROJECT_DIR}')

# Install Wayfinder deps
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', PROJECT_DIR],
    capture_output=True, text=True
)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
if LINTGATE_DIR not in sys.path:
    sys.path.insert(0, LINTGATE_DIR)

for m in list(sys.modules):
    if any(m.startswith(d.split('/')[0]) for d in SRC_DIRS) or m.startswith('lintgate'):
        del sys.modules[m]
importlib.invalidate_caches()

from lintgate.specification.mutation_engine import MutationCategory
print(f'Import OK. Categories: {[c.value for c in MutationCategory]}')
print('Ready!')

## Step 2: Mutation sweep


In [ ]:
import ast, json, os, sys, time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

WORKERS = 4
BUDGET_MS = 1000.0

def collect_python_files(root):
    files = []
    for subdir in SRC_DIRS:
        base = os.path.join(root, subdir)
        if not os.path.isdir(base):
            continue
        for dirpath, _, filenames in os.walk(base):
            for fn in sorted(filenames):
                if fn.endswith('.py') and not fn.startswith('test_') and not fn.endswith('_test.py'):
                    files.append(os.path.relpath(os.path.join(dirpath, fn), root))
    return files

def profile_single_file(args):
    project_root, rel_path, budget_ms = args
    import ast, os, sys, time
    sys.path.insert(0, project_root)
    from lintgate.specification.mutation_engine import run_function_sampling
    from lintgate.keys import canonical_function_key
    from lintgate.specification.mutation_filter import filter_categories
    from mcp_tools._mutation_impl import (
        MutationContext, detect_purity_map, discover_test_files,
        get_cache_dir, load_test_callables, lookup_purity,
        parse_file, save_cached_state, walk_functions,
    )
    full_path = os.path.join(project_root, rel_path)
    cache_dir = get_cache_dir(project_root)
    start = time.monotonic()
    result = {'file': rel_path, 'profiled': 0, 'trivial': 0, 'killed': 0, 'survived': 0, 'errors': 0}
    tree = parse_file(full_path)
    if tree is None:
        result['errors'] = 1
        return result
    functions = walk_functions(tree)
    if not functions:
        return result
    test_files = discover_test_files(project_root, full_path)
    purity_map = detect_purity_map(full_path)
    ctx = MutationContext(
        full_path=full_path, rel_path=rel_path, cache_dir=cache_dir,
        purity_map=purity_map, test_files=test_files, project_root=project_root,
    )
    for qualname, node in functions:
        func_key = canonical_function_key(rel_path, qualname)
        body = getattr(node, 'body', [])
        if len(body) <= 1:
            stmt = body[0] if body else None
            if isinstance(stmt, (ast.Return, ast.Expr)):
                result['trivial'] += 1
                continue
        is_pure = lookup_purity(purity_map, qualname)
        cats = filter_categories(node, is_pure=is_pure)
        bare_name = qualname.split('.')[-1]
        tests, diag = load_test_callables(
            ctx.test_files, bare_name,
            project_root=ctx.project_root, func_key=func_key,
        )
        try:
            sr = run_function_sampling(
                node, func_key, cats, tests, lambda *_: None, budget_ms=budget_ms,
            )
            rd = sr.to_dict()
            rd['tests_loaded'] = len(tests)
            rd['is_pure'] = is_pure
            args_node = getattr(node, 'args', None)
            rd['parameter_count'] = len(args_node.args) if args_node else 0
            if diag:
                from dataclasses import asdict as _asdict
                rd['discovery_diagnostics'] = _asdict(diag) if hasattr(diag, '__dataclass_fields__') else diag
            if not ctx.test_files:
                ds = 'NO_TEST_FILES'
            elif len(tests) == 0:
                ds = 'NO_TESTS_LINKED'
            elif sr.total_killed == 0 and sr.total_mutants == 0:
                ds = 'EQUIVALENT'
            elif sr.total_killed == 0 and sr.total_mutants > 0:
                ds = 'ZERO_KILLS'
            else:
                ds = 'OK'
            rd['discovery_state'] = ds
            save_cached_state(ctx.cache_dir, func_key, rd)
            result['profiled'] += 1
            result['killed'] += sr.total_killed
            result['survived'] += sr.total_survived
        except Exception:
            result['errors'] += 1
    result['elapsed_s'] = round(time.monotonic() - start, 2)
    return result

project_root = os.path.abspath(PROJECT_DIR)
files = collect_python_files(project_root)
print(f'Found {len(files)} source files | Workers: {WORKERS} | Budget: {BUDGET_MS}ms')
cache_dir = Path(project_root) / '.lintgate' / 'mutation'
cache_dir.mkdir(parents=True, exist_ok=True)
work = [(project_root, f, BUDGET_MS) for f in files]
totals = {'profiled': 0, 'trivial': 0, 'killed': 0, 'survived': 0, 'errors': 0}
start = time.monotonic()
with ProcessPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(profile_single_file, w): w[1] for w in work}
    for i, future in enumerate(as_completed(futures), 1):
        rel = futures[future]
        try:
            r = future.result()
            for k in totals:
                totals[k] += r.get(k, 0)
            if i % 50 == 0 or i == len(files):
                print(f'[{i}/{len(files)}] {totals["profiled"]} profiled, {totals["killed"]}k/{totals["survived"]}s')
        except Exception as e:
            print(f'[{i}/{len(files)}] {rel}: FAILED - {e}')
            totals['errors'] += 1
elapsed = round(time.monotonic() - start, 1)
t = totals['killed'] + totals['survived']
kr = f'{totals["killed"]}/{t} ({totals["killed"]/t:.1%})' if t else 'N/A'
print(f'\n{"="*60}')
print(f'Sweep done in {elapsed}s | Kill rate: {kr}')
print(f'Profiled: {totals["profiled"]} | Trivial: {totals["trivial"]} | Errors: {totals["errors"]}')


## Step 3: Specification analysis + reconciliation


In [ ]:
import json, os
from pathlib import Path
from collections import Counter

project_root = os.path.abspath(PROJECT_DIR)
cache_dir = Path(project_root) / '.lintgate' / 'mutation'
profiles = {}
for f in cache_dir.glob('*.json'):
    if f.name in ('sweep_summary.json', 'scheduler_state.json'):
        continue
    try:
        d = json.loads(f.read_text())
        profiles[d.get('function_key', '')] = d
    except Exception:
        pass

from lintgate.specification.file_analyzer import analyze_file
from lintgate.specification.static_empirical_reconciliation import build_overlay, reconcile_spec_level
from lintgate.specification.gap_classifier import classify_from_func_data

files = collect_python_files(project_root)
gap_dist, regime_dist, phase_dist = Counter(), Counter(), Counter()
under_specified = []
total_funcs, total_spec, total_reconciled = 0, 0.0, 0.0

for rel_file in files:
    try:
        result = analyze_file(project_root, rel_file)
    except Exception:
        continue
    if not result or not result.functions:
        continue
    for func_key, func_data in result.functions.items():
        if not isinstance(func_data, dict):
            continue
        total_funcs += 1
        sigma = func_data.get('sigma', 0) or 0
        regime = func_data.get('regime', 'A')
        phase = func_data.get('phase', 'bulk')
        static_spec = func_data.get('specification_level', 0.0)
        regime_dist[regime] += 1
        phase_dist[phase] += 1
        total_spec += static_spec
        overlay = build_overlay(func_key, int(sigma), regime, phase, profiles or None)
        reconciled, source = reconcile_spec_level(static_spec, overlay)
        total_reconciled += reconciled
        mutation_entry = profiles.get(func_key)
        gap = classify_from_func_data(func_data, mutation_entry).value
        gap_dist[gap] += 1
        kill_rate = 1.0 - mutation_entry.get('survival_rate', 1.0) if mutation_entry else None
        if reconciled < 0.5 and sigma > 3:
            under_specified.append((rel_file, func_key, static_spec, reconciled, kill_rate, sigma))

ms = total_spec / total_funcs if total_funcs else 0
mr = total_reconciled / total_funcs if total_funcs else 0
print(f'Functions: {total_funcs} | spec_level: {ms:.3f} static / {mr:.3f} reconciled')
print(f'Regime: {dict(regime_dist.most_common())}')  
print(f'Phase: {dict(phase_dist.most_common())}')
print(f'\nGap classification:')
for gap, n in gap_dist.most_common():
    print(f'  {gap}: {n}')
under_specified.sort(key=lambda x: -(x[5] or 0))
print(f'\nUnder-specified (reconciled < 0.5, sigma > 3): {len(under_specified)}')
for f, fk, ss, rs, kr, sig in under_specified[:20]:
    kr_s = f'{kr:.0%}' if kr is not None else '?'
    print(f'  sigma={sig:>3} spec={rs:.2f} kill={kr_s} {fk}')


## Step 4: Per-file kill rates


In [ ]:
import json
from pathlib import Path
from collections import Counter

cache_dir = Path(PROJECT_DIR) / '.lintgate' / 'mutation'
pfiles = [f for f in cache_dir.glob('*.json') if f.name not in ('sweep_summary.json', 'scheduler_state.json')]
total_k, total_s = 0, 0
per_file = {}
for f in pfiles:
    try:
        d = json.loads(f.read_text())
    except Exception:
        continue
    k, s = d.get('total_killed', 0), d.get('total_survived', 0)
    total_k += k
    total_s += s
    fk = d.get('function_key', '?')
    src = fk.split('::')[0] if '::' in fk else '?'
    if src not in per_file:
        per_file[src] = {'killed': 0, 'survived': 0, 'funcs': 0}
    per_file[src]['killed'] += k
    per_file[src]['survived'] += s
    per_file[src]['funcs'] += 1
t = total_k + total_s
print(f'Aggregate: {total_k}/{t} ({total_k/t:.1%})' if t else 'No data')
rated = []
for src, d in per_file.items():
    tt = d['killed'] + d['survived']
    kr = d['killed'] / tt if tt > 0 else -1
    rated.append((src, d['killed'], d['survived'], tt, kr, d['funcs']))
rated.sort(key=lambda x: x[4])
fk_count = sum(1 for r in rated if r[4] == 1.0)
zk_count = sum(1 for r in rated if r[4] == 0.0 and r[3] > 0)
print(f'{fk_count} full-kill | {len(rated)-fk_count-zk_count} partial | {zk_count} zero-kill')
print(f'\n--- Zero-kill ---')
for src, k, s, tt, kr, funcs in rated:
    if kr == 0.0 and tt > 0:
        print(f'  {tt:>4} mutants  {funcs:>3} funcs  {src}')
print(f'\n--- Partial (worst 15) ---')
partial = [(src, k, s, tt, kr) for src, k, s, tt, kr, _ in rated if 0 < kr < 1.0]
for src, k, s, tt, kr in partial[:15]:
    print(f'  {kr:>5.0%} ({k}/{tt})  {src}')
summary = {'profiles': len(pfiles), 'killed': total_k, 'survived': total_s,
           'kill_rate': round(total_k/t, 4) if t else 0, 'full_kill': fk_count, 'zero_kill': zk_count}
with open(cache_dir / 'sweep_summary.json', 'w') as sf:
    json.dump(summary, sf, indent=2)
print(f'\nSummary written.')


## Step 5: Download results


In [ ]:
import os, subprocess
from pathlib import Path

mutation_dir = Path(PROJECT_DIR) / '.lintgate' / 'mutation'
mutation_dir.mkdir(parents=True, exist_ok=True)

zip_path = '/content/golden_path_results.zip'
subprocess.run(['zip', '-r', zip_path, '.lintgate/mutation/'],
               cwd=PROJECT_DIR, capture_output=True)

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'Size: {size_mb:.1f} MB')
    try:
        from google.colab import files
        files.download(zip_path)
        print('Download started!')
    except ImportError:
        print(f'Not in Colab. Results at: {zip_path}')
else:
    print('No zip created.')

print(f'\nTo apply locally: unzip ~/Downloads/golden_path_results.zip -d /path/to/Wayfinder/')